# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [3]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

# Create a virtual environment
!uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

/bin/bash: line 1: uv: command not found


### Run the cell below every time to activate the installed environment. 

In [ ]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [8]:
print("Task started...")

import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

print("Task complete!")

Task started...
Task complete!


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [9]:
print("Task started...")

data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

print("Task complete!")

Task started...
Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}
Task complete!


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [10]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# tokenizer.pad_token = tokenizer.eos_token

# llm = LLM(
#     model=MODEL_ID,
#     quantization="bitsandbytes",
#     load_format="bitsandbytes",
#     enable_prefix_caching=False,
#     gpu_memory_utilization=0.50,
#     max_model_len=16384,
#     trust_remote_code=True,
#     max_num_seqs=256,
#     max_num_batched_tokens=32768,
# )

# sampling_params = SamplingParams(
#     max_tokens=MAX_TOKENS,
#     temperature=0.6,
#     top_p=0.95,
#     top_k=20,
#     min_p=0.0,
#     presence_penalty=0.0,
#     repetition_penalty=1.0,
# )

# print("Model loaded.")

## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [ ]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Generate
# print(f"Generating responses for {len(prompts)} questions...")
# outputs = llm.generate(prompts, sampling_params=sampling_params)

# responses = [out.outputs[0].text.strip() for out in outputs]

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

### Generate with Transformers (for Datahub)

In [14]:
from transformers import TextStreamer

import torch
print(torch.cuda.is_available(), "\n")
print(torch.cuda.get_device_name(0), "\n")
print(next(llm.parameters()).device, "\n")

# Build prompts for first 5 entries
prompts = []
for item in data[:5]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

responses = []

for i, prompt in enumerate(prompts):
    print(f"\n── Generating Response {i} (id={data[i].get('id')}) ──")

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=16384,
    ).to(llm.device)

    streamer = TextStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            temperature=0.6,
            top_p=0.95,
            top_k=20,
            repetition_penalty=1.0,
            do_sample=True,
            streamer=streamer,
        )

    # Decode only new tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    responses.append(response)

    print(f"\n── Finished Response {i} ──")

True 

NVIDIA A30 MIG 2g.12gb 

cuda:0 


── Generating Response 0 (id=0) ──
Okay, let's see. I need to find the sum of the first 325 positive even whole numbers. Hmm, positive even whole numbers start at 2, right? Because the first positive even number is 2, then 4, 6, 8, and so on. Wait, actually, sometimes people might think of 0 as a whole number, but the problem says "positive even whole numbers," so 0 is not positive. So the first one is 2, then 4, ื่, up to the 325th one. 

First, let's recall that the sum of the first n even numbers. Wait, but here it's the first 325 positive even whole numbers. Let me think. The sequence is 2, 4, 6, 8, ..., up to the 325th term. 

This is an arithmetic sequence where the first term a₁ = 2, the common difference d = 2, and we need the sum of the first 325 terms. 

The formula for the sum of the first n terms of an arithmetic sequence is Sₙ = n/2 * [2a₁ + (n - 1)d] or Sₙ = n*(a₁ + aₙ)/2, where aₙ is the nth term. 

Let me try to figure out the 3

KeyboardInterrupt: 

In [13]:
# Build prompts for first 5 entries
prompts = []
for item in data[:5]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Tokenize (padded batch)
print(f"Generating responses for {len(prompts)} questions...")
inputs = tokenizer(
    prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=16384,
).to(llm.device)

# Generate
with torch.inference_mode():
    output_ids = llm.generate(
        **inputs,
        max_new_tokens=MAX_TOKENS,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        do_sample=True,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

# Decode only the new tokens (strip the prompt)
responses = []
for i, out in enumerate(output_ids):
    new_tokens = out[inputs["input_ids"].shape[1]:]
    responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Generating responses for 5 questions...


RuntimeError: NVML_SUCCESS == r INTERNAL ASSERT FAILED at "../c10/cuda/CUDACachingAllocator.cpp":995, please report a bug to PyTorch. 

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [11]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   0%|          | 5/1126 [00:00<01:31, 12.31it/s]

Scoring complete. 5 results.


## 8. Summary

Print accuracy broken down by question type.

In [12]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    1 /    2  (50.00%)
  Free-form  :    2 /    3  (66.67%)
  Overall    :    3 /    5  (60.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [13]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 5 records to results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!

# Running the CSE 151B Competition Notebook from a Fresh Kernel

## 0. Open a JupyterLab Terminal

In JupyterLab, open:

```txt
File → New → Terminal
```

Then run:

```bash
cd ~/151B_SP26_Competition
source .venv/bin/activate
```

You should see `(.venv)` at the front of your terminal prompt.

---

## 1. Open the Notebook

Open:

```txt
starter_code_cse151b_comp.ipynb
```

Then make sure the kernel is using the project environment:

```txt
Kernel → Change Kernel → Python (cse151b)
```

If you do not see `Python (cse151b)`, run this in the terminal:

```bash
cd ~/151B_SP26_Competition
source .venv/bin/activate
python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"
```

Then refresh JupyterLab and check the kernel list again.

---

## 2. Run Cell 1: Imports, Config, Data, Prompts

Run the cell labeled:

```txt
Cell 1: imports, config, data, prompts
```

This cell:

- imports required packages
- sets the GPU ID
- loads the dataset from `data/public.jsonl`
- sets output paths
- sets batch sizes and token limits
- defines MCQ and FRQ prompts
- defines prompt-building functions

Before running, choose your run name:

```python
RUN_NAME = "run1"
```

This controls where responses and scores are saved:

```txt
results/run1_responses.jsonl
results/run1_eval.jsonl
results/run1_summary.json
```

To start a new run, change it:

```python
RUN_NAME = "run2"
```

To resume an old run, set it back to the old name:

```python
RUN_NAME = "run1"
```

---

## 3. Run Cell 2: Tokenizer and Model

Run the cell labeled:

```txt
Cell 2: loading tokenizer and model
```

This loads:

- tokenizer
- Qwen model
- 4-bit quantization config
- GPU placement

You should see output like:

```txt
CUDA available: True
GPU: NVIDIA A30
Model device: cuda:0
```

If `Model device` says `cpu`, stop and fix the kernel/GPU setup before continuing.

To monitor GPU usage from terminal:

```bash
watch -n 1 nvidia-smi
```

On DSMLP/MIG, `GPU-Util` may show `N/A` and the process table may be empty. That is normal. Look for GPU memory usage, temperature, and power usage instead.

---

## 4. Run Cell 3: Response Generation with Checkpointing

Run the cell labeled:

```txt
Cell 3: generating responses with checkpointing
```

This cell generates model responses and saves progress after every completed batch.

It writes to:

```txt
results/<RUN_NAME>_responses.jsonl
```

For example:

```txt
results/run1_responses.jsonl
```

If the notebook crashes, disconnects, or gets interrupted, completed responses are already saved.

To resume:

1. Restart the kernel if needed.
2. Rerun Cell 1.
3. Rerun Cell 2.
4. Rerun Cell 3.

It will load the existing response file and continue where it left off.

You should see something like:

```txt
Already completed: 344/1126
```

That means it found saved progress and will skip already completed questions.

---

## 5. Token and Batch Settings

The important settings are in Cell 1:

```python
MAX_TOKENS_MCQ = 64
MAX_TOKENS_FRQ = 512

BATCH_SIZE_FRQ = 1
BATCH_SIZE_MCQ = max(1, BATCH_SIZE_FRQ * (MAX_TOKENS_FRQ // MAX_TOKENS_MCQ))
BATCH_SIZE_MCQ = min(BATCH_SIZE_MCQ, 8)
```

The idea is:

```txt
MCQ needs fewer tokens → larger batch
FRQ needs more tokens → smaller batch
```

Example:

```txt
MAX_TOKENS_MCQ = 64
MAX_TOKENS_FRQ = 512
```

Since `512 / 64 = 8`, the MCQ batch size is about 8 times larger than the FRQ batch size.

If generation is too slow, lower:

```python
MAX_TOKENS_FRQ = 256
```

If answers are being cut off, raise:

```python
MAX_TOKENS_FRQ = 768
```

If CUDA crashes, lower batch sizes:

```python
BATCH_SIZE_FRQ = 1
BATCH_SIZE_MCQ = 2
```

---

## 6. Run Cell 4: Evaluation and Score Saving

After generation finishes, run the cell labeled:

```txt
Cell 4: evaluation and score saving
```

This cell:

- loads saved responses from `results/<RUN_NAME>_responses.jsonl`
- scores MCQ answers
- uses `Judger` for free-response questions
- saves full evaluation results
- saves a summary JSON

Outputs:

```txt
results/run1_eval.jsonl
results/run1_summary.json
```

It will print something like:

```txt
EVALUATION RESULTS
MCQ       : ...
FRQ       : ...
Overall   : ...
With boxed answer    : ...
Without boxed answer : ...
Missing responses    : ...
```

If there are missing responses, rerun Cell 3 before trusting the final score.

---

## 7. Typical Fresh-Kernel Workflow

Every time you come back from a fresh kernel:

In terminal:

```bash
cd ~/151B_SP26_Competition
source .venv/bin/activate
```

Then in the notebook:

```txt
1. Select kernel: Python (cse151b)
2. Run Cell 1
3. Run Cell 2
4. Run Cell 3 to generate/resume
5. Run Cell 4 to evaluate/save scores
```

---

## 8. Important Notes

Do not rerun the old install cell unless packages are missing.

Do not delete your response file unless you want to restart from zero:

```txt
results/<RUN_NAME>_responses.jsonl
```

To start a clean run, change:

```python
RUN_NAME = "run2"
```

instead of deleting old results.

If the generation cell gets interrupted, rerun it. It will resume from the saved file.

If CUDA gives an internal allocator error, restart the kernel or restart the DataHub server, then rerun Cells 1–3. Your completed responses should still be saved.


In [5]:
print("Cell 1: imports, config, data, prompts...")

import os
import re
import sys
import gc
import json
import time
import shutil
from pathlib import Path
from typing import Optional

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ── Main run config ─────────────────────────────────────────────
RUN_NAME = "run7"   # CHANGE THIS to resume/save from a different file

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH = "data/public.jsonl"

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RESPONSE_PATH = RESULTS_DIR / f"{RUN_NAME}_responses.jsonl"
EVAL_PATH = RESULTS_DIR / f"{RUN_NAME}_eval.jsonl"
SUMMARY_PATH = RESULTS_DIR / f"{RUN_NAME}_summary.json"

# ── GPU config ─────────────────────────────────────────────────
GPU_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

# ── Generation config ──────────────────────────────────────────
MAX_INPUT_LEN = 16384

# Thinking model needs enough room to reach final answer
MAX_TOKENS_MCQ = 4096
MAX_TOKENS_FRQ = 4096*2

# Keep conservative to avoid CUDA/NVML crashes.
BATCH_SIZE_FRQ = 1
BATCH_SIZE_MCQ = 2

print(f"RUN_NAME        = {RUN_NAME}")
print(f"RESPONSE_PATH   = {RESPONSE_PATH}")
print(f"EVAL_PATH       = {EVAL_PATH}")
print(f"MAX_INPUT_LEN   = {MAX_INPUT_LEN}")
print(f"MCQ batch/toks  = {BATCH_SIZE_MCQ}, {MAX_TOKENS_MCQ}")
print(f"FRQ batch/toks  = {BATCH_SIZE_FRQ}, {MAX_TOKENS_FRQ}")

# ── Load data ──────────────────────────────────────────────────
t0 = time.time()

with open(DATA_PATH, "r") as f:
    data = [json.loads(line) for line in f]

# Debug
data = data[:16]

n_mcq = sum(bool(d.get("options")) for d in data)
n_frq = len(data) - n_mcq

print(f"Loaded {len(data)} questions: {n_mcq} MCQ, {n_frq} FRQ")
print(f"Data loading took {time.time() - t0:.2f}s")

# ── Prompts ────────────────────────────────────────────────────
# Qwen3-4B-Thinking-2507 is thinking-only, so do NOT rely on disabling thinking.
# We will parse away the thinking content after generation.

SYSTEM_PROMPT_MCQ = (
    "You are solving a multiple-choice math problem. "
    "Reason efficiently and avoid unnecessary repetition. "
    "At the end, output exactly one final answer in the format \\boxed{A}. "
    "The boxed answer must contain only the choice letter."
)

SYSTEM_PROMPT_FRQ = (
    "You are solving a math problem. "
    "Reason efficiently and avoid unnecessary repetition. "
    "At the end, output exactly one final answer in the format \\boxed{...}. "
    "Do not add text after the boxed answer."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(
            f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options)
        )
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"

    return SYSTEM_PROMPT_FRQ, question

def make_prompt(item: dict) -> str:
    system, user = build_prompt(item["question"], item.get("options"))

    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

print("Cell 1 complete.")

Cell 1: imports, config, data, prompts...
RUN_NAME        = run7
RESPONSE_PATH   = results/run7_responses.jsonl
EVAL_PATH       = results/run7_eval.jsonl
MAX_INPUT_LEN   = 16384
MCQ batch/toks  = 2, 4096
FRQ batch/toks  = 1, 8192
Loaded 16 questions: 7 MCQ, 9 FRQ
Data loading took 0.01s
Cell 1 complete.


In [6]:
print("Cell 2: loading tokenizer and model...")

t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print(f"Tokenizer loaded in {time.time() - t0:.2f}s")

t0 = time.time()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)

llm.eval()

print(f"Model loaded in {(time.time() - t0) / 60:.2f} min")

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Allocated GB:", torch.cuda.memory_allocated() / 1024**3)
    print("Reserved GB:", torch.cuda.memory_reserved() / 1024**3)

model_device = next(llm.parameters()).device
print("Model device:", model_device)

assert torch.cuda.is_available(), "CUDA is not available."
assert "cuda" in str(model_device), f"Model is not on GPU. Device = {model_device}"

print("Cell 2 complete.")

Cell 2: loading tokenizer and model...
Tokenizer loaded in 0.97s


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded in 0.04 min
CUDA available: True
GPU: NVIDIA A30 MIG 2g.12gb
Allocated GB: 5.082736968994141
Reserved GB: 5.142578125
Model device: cuda:0
Cell 2 complete.


In [7]:
print("Cell 3: generating responses with checkpointing...")

gc.collect()
torch.cuda.empty_cache()

print("Using response file:", RESPONSE_PATH)

# ── Load existing responses from selected run file ──────────────
responses_by_id = {}

if RESPONSE_PATH.exists():
    with open(RESPONSE_PATH, "r") as f:
        for line in f:
            row = json.loads(line)
            responses_by_id[str(row["id"])] = row["response"]

print(f"Already completed: {len(responses_by_id)}/{len(data)}")

# ── Atomic checkpoint save ─────────────────────────────────────
def save_responses_atomic():
    tmp_path = RESPONSE_PATH.with_suffix(".tmp")
    backup_path = RESPONSE_PATH.with_suffix(".bak")

    with open(tmp_path, "w") as f:
        for item in data:
            qid = str(item["id"])
            if qid in responses_by_id:
                f.write(json.dumps({
                    "id": item["id"],
                    "is_mcq": bool(item.get("options")),
                    "response": responses_by_id[qid],
                }) + "\n")

    if RESPONSE_PATH.exists():
        shutil.copy2(RESPONSE_PATH, backup_path)

    tmp_path.replace(RESPONSE_PATH)

def format_eta(seconds):
    if seconds < 60:
        return f"{seconds:.0f}s"
    if seconds < 3600:
        return f"{seconds / 60:.1f}m"
    return f"{seconds / 3600:.1f}h"

THINK_END_TOKEN_ID = 151668

def extract_qwen_final_content(output_token_ids):
    """
    Qwen3-Thinking generates reasoning first, then </think>, then final answer.
    If </think> appears, keep only content after it.
    If not, keep full output as fallback.
    """
    ids = output_token_ids.tolist() if hasattr(output_token_ids, "tolist") else list(output_token_ids)

    try:
        idx = len(ids) - ids[::-1].index(THINK_END_TOKEN_ID)
        final_ids = ids[idx:]
        final_text = tokenizer.decode(final_ids, skip_special_tokens=True).strip()
        if final_text:
            return final_text
    except ValueError:
        pass

    return tokenizer.decode(ids, skip_special_tokens=True).strip()

# ── Generation helper ──────────────────────────────────────────
def generate_group(items, batch_size, max_tokens, label):
    remaining = [item for item in items if str(item["id"]) not in responses_by_id]

    print(f"\n{label}: {len(items) - len(remaining)}/{len(items)} already done")
    print(f"{label}: {len(remaining)} remaining")
    print(f"{label}: batch_size={batch_size}, max_tokens={max_tokens}")

    if not remaining:
        return

    group_start_time = time.time()
    pbar = tqdm(range(0, len(remaining), batch_size), desc=label)

    for batch_start in pbar:
        batch = remaining[batch_start:batch_start + batch_size]

        try:
            t_prompt = time.time()

            prompts = [make_prompt(item) for item in batch]

            inputs = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_INPUT_LEN,
            ).to(llm.device)

            input_len = inputs["input_ids"].shape[1]
            prompt_time = time.time() - t_prompt

            t_gen = time.time()

            with torch.inference_mode():
                output_ids = llm.generate(
                    **inputs,
                    max_new_tokens=max_tokens,
                    do_sample=False,
                    repetition_penalty=1.0,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            gen_time = time.time() - t_gen

            t_decode = time.time()

            for item, out in zip(batch, output_ids):
                new_tokens = out[input_len:]
                response = extract_qwen_final_content(new_tokens)
                responses_by_id[str(item["id"])] = response

            decode_time = time.time() - t_decode

            save_responses_atomic()

            completed_total = len(responses_by_id)
            completed_group = min(batch_start + len(batch), len(remaining))
            elapsed_group = time.time() - group_start_time

            avg_sec_per_remaining_item = elapsed_group / max(completed_group, 1)
            eta_group = avg_sec_per_remaining_item * (len(remaining) - completed_group)

            pbar.set_postfix({
                "total_done": f"{completed_total}/{len(data)}",
                "prompt_s": f"{prompt_time:.1f}",
                "gen_s": f"{gen_time:.1f}",
                "decode_s": f"{decode_time:.1f}",
                "ETA": format_eta(eta_group),
            })

        except Exception as e:
            print("\nGeneration crashed. Saving completed responses before raising error...")
            save_responses_atomic()
            print(f"Saved {len(responses_by_id)} responses to {RESPONSE_PATH}")
            raise e

        finally:
            try:
                del inputs, output_ids
            except NameError:
                pass

            gc.collect()
            torch.cuda.empty_cache()

# ── Split MCQ and FRQ into separate groups ──────────────────────
mcq_items = [item for item in data if item.get("options")]
frq_items = [item for item in data if not item.get("options")]

total_start_time = time.time()

# MCQ first: cheap and fast
generate_group(
    items=mcq_items,
    batch_size=BATCH_SIZE_MCQ,
    max_tokens=MAX_TOKENS_MCQ,
    label="MCQ",
)

# FRQ second: slower and larger generations
generate_group(
    items=frq_items,
    batch_size=BATCH_SIZE_FRQ,
    max_tokens=MAX_TOKENS_FRQ,
    label="FRQ",
)

# Build ordered lists for evaluation compatibility
responses = [responses_by_id[str(item["id"])] for item in data if str(item["id"]) in responses_by_id]
response_ids = [item["id"] for item in data if str(item["id"]) in responses_by_id]

print("\nGeneration cell complete.")
print(f"Generated/saved {len(responses_by_id)}/{len(data)} responses.")
print(f"Saved to: {RESPONSE_PATH}")
print(f"Total generation time this run: {format_eta(time.time() - total_start_time)}")

if len(responses_by_id) < len(data):
    print("Not all responses are complete yet. Rerun this cell to resume.")
else:
    print("All responses complete.")

Cell 3: generating responses with checkpointing...
Using response file: results/run7_responses.jsonl
Already completed: 0/16

MCQ: 0/7 already done
MCQ: 7 remaining
MCQ: batch_size=2, max_tokens=4096


MCQ:   0%|          | 0/4 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
print("Cell 4: evaluation and score saving...")

t0 = time.time()

# ── Load responses from selected run file ───────────────────────
responses_by_id = {}

with open(RESPONSE_PATH, "r") as f:
    for line in f:
        row = json.loads(line)
        responses_by_id[str(row["id"])] = row["response"]

print(f"Loaded {len(responses_by_id)}/{len(data)} responses from {RESPONSE_PATH}")

missing = [item["id"] for item in data if str(item["id"]) not in responses_by_id]
if missing:
    print(f"Warning: {len(missing)} responses missing. First few missing ids: {missing[:10]}")

# ── Judger setup ───────────────────────────────────────────────
sys.path.insert(0, ".")
from judger import Judger

judger = Judger(strict_extract=False)

def extract_boxed(text: str) -> str:
    matches = re.findall(r"\\boxed\{([^{}]*)\}", text)
    return matches[-1].strip() if matches else ""

def extract_letter(text: str) -> str:
    # First try JSON-style answer field
    m = re.search(r'"answer"\s*:\s*"([A-Za-z])"', text)
    if m:
        return m.group(1).upper()

    # Then try boxed letter
    boxed = extract_boxed(text)
    if re.fullmatch(r"[A-Za-z]", boxed):
        return boxed.upper()

    # Last fallback: any standalone capital letter
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

def has_boxed(text: str) -> bool:
    return bool(re.search(r"\\boxed\{.*?\}", text, flags=re.DOTALL))

# ── Evaluate ───────────────────────────────────────────────────
results = []

eval_start = time.time()

for item in tqdm(data, desc="Scoring"):
    qid = str(item["id"])

    if qid not in responses_by_id:
        continue

    response = responses_by_id[qid]
    is_mcq = bool(item.get("options"))
    gold = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]

        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id": item["id"],
        "is_mcq": is_mcq,
        "gold": gold,
        "response": response,
        "has_boxed": has_boxed(response),
        "correct": correct,
    })

eval_time = time.time() - eval_start

# ── Save eval results ──────────────────────────────────────────
with open(EVAL_PATH, "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

# ── Summary ────────────────────────────────────────────────────
mcq_res = [r for r in results if r["is_mcq"]]
frq_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

summary = {
    "run_name": RUN_NAME,
    "response_path": str(RESPONSE_PATH),
    "eval_path": str(EVAL_PATH),
    "num_total_data": len(data),
    "num_evaluated": len(results),
    "num_missing": len(missing),
    "mcq_correct": sum(r["correct"] for r in mcq_res),
    "mcq_total": len(mcq_res),
    "mcq_accuracy": acc(mcq_res),
    "frq_correct": sum(r["correct"] for r in frq_res),
    "frq_total": len(frq_res),
    "frq_accuracy": acc(frq_res),
    "overall_correct": sum(r["correct"] for r in results),
    "overall_total": len(results),
    "overall_accuracy": acc(results),
    "num_with_boxed": sum(r["has_boxed"] for r in results),
    "num_without_boxed": sum(not r["has_boxed"] for r in results),
    "eval_time_sec": eval_time,
}

with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"MCQ       : {summary['mcq_correct']:4d} / {summary['mcq_total']:4d} = {summary['mcq_accuracy']:.2f}%")
print(f"FRQ       : {summary['frq_correct']:4d} / {summary['frq_total']:4d} = {summary['frq_accuracy']:.2f}%")
print(f"Overall   : {summary['overall_correct']:4d} / {summary['overall_total']:4d} = {summary['overall_accuracy']:.2f}%")
print("-" * 50)
print(f"With boxed answer    : {summary['num_with_boxed']}")
print(f"Without boxed answer : {summary['num_without_boxed']}")
print(f"Missing responses    : {summary['num_missing']}")
print("-" * 50)
print(f"Saved eval to    : {EVAL_PATH}")
print(f"Saved summary to : {SUMMARY_PATH}")
print(f"Eval time        : {eval_time:.2f}s")
print("=" * 50)

print("Cell 4 complete.")